# **Cost Benefit Analysis**
---
This is where we'll turn our model's predictions from "the model has 83% recall" into a business case with real dollar figures with results like "targeting these specific customers saves the business $X net of retention costs." 

In this notebook, I will:
1. load predictions and set business assumptions
2. segment the flagged customers
3. calculate ROI per tier
4. then export the retention recommendations

### **Imports and loading data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/xgb_predictions.csv')
print(df.shape)
df.head()

### **Business Assumptions**

These are what will basically control the analysis based on the dataset averages and standard retention campaign costs

In [ ]:
MONTHLY_REVENUE = 64.76  # average MonthlyCharges from the dataset
RETENTION_OFFER_COST = 15.00  # cost of one retention offer (discount/incentive)
RETENTION_SUCCESS_RATE = 0.30  # industry standard benchmark: 30% of at-risk customers accept and stay
CUSTOMER_LIFETIME = 12   # months estimmate of retained value

print(f"Average monthly revenue per customer: ${MONTHLY_REVENUE}")
print(f"Retention offer cost: ${RETENTION_OFFER_COST}")
print(f"Assumed retention success rate: {RETENTION_SUCCESS_RATE*100}%")
print(f"Customer lifetime assumption: {CUSTOMER_LIFETIME} months")

# note that these can be challenged and adjusted by the customer but it's important to be clear about them from the start with proper justifications.

### **Isolating flagged customers and ranking by risk**
---
Of the 601 customers that were flagged (check modelling.ipynb), who are the highest priority?

What we'll do is to rank them by **churn probability score** and **create tiers** say top 100, top 200, top 300, so that a *business can decide how much retention budget to deploy.*

In [ ]:
# looking at only customers the model flagged as churn risk
flagged = df[df['Predicted_Churn'] == 1].copy()
flagged = flagged.sort_values('Churn_Probability', ascending=False).reset_index(drop=True)

print(f"Total flagged: {len(flagged)}")
print(f"Of which are real churners: {flagged['Actual_Churn'].sum()}")
print(f"\nProbability score distribution:")
print(flagged['Churn_Probability'].describe().round(3))

### **Assigning risk tiers**
---
Here is where we do the actual segmentation of the 601 flagged customers into **three buckets by confidence level.** 

High risk customers should be contacted first as they're the most likely to leave and the most urgent to act on.

In [ ]:
def assign_tier(prob):
    if prob >= 0.70:
        return 'High risk'
    elif prob >= 0.50:
        return 'Medium risk'
    else:
        return 'Low risk'

flagged['Risk_Tier'] = flagged['Churn_Probability'].apply(assign_tier)

print(flagged['Risk_Tier'].value_counts())

### **Calculating the Return on Investment (ROI) per risk tier**
---
*why?*
Because for each tier calculated, we need to know:
- what does it cost to run the campaign?
- how many customers will we actually retain (flagged × real churners × 30% success rate)?
- and what's the net dollar value after subtracting offer costs?

So the **ROI tells us which tier gives the best return per dollar spent.**

In [ ]:
tier_results = []

for tier in ['High risk', 'Medium risk', 'Low risk']:
    group = flagged[flagged['Risk_Tier'] == tier]
    n_customers = len(group)
    n_real_churners = group['Actual_Churn'].sum()
    
    campaign_cost = n_customers * RETENTION_OFFER_COST
    customers_saved = n_real_churners * RETENTION_SUCCESS_RATE
    revenue_saved = customers_saved * MONTHLY_REVENUE * CUSTOMER_LIFETIME
    net_value = revenue_saved - campaign_cost
    roi = (net_value / campaign_cost * 100) if campaign_cost > 0 else 0
    
    tier_results.append({
        'Risk Tier': tier,
        'Customers Flagged': n_customers,
        'Actual Churners': int(n_real_churners),
        'Campaign Cost ($)': round(campaign_cost, 2),
        'Est. Revenue Saved ($)': round(revenue_saved, 2),
        'Net Value ($)': round(net_value, 2),
        'ROI (%)': round(roi, 1)
    })

roi_df = pd.DataFrame(tier_results)
print(roi_df.to_string(index=False))

### **Putting all the results together so far:**

In [ ]:
total_cost = roi_df['Campaign Cost ($)'].sum()
total_saved = roi_df['Estimated Revenue Saved ($)'].sum()
total_net = roi_df['Net Value ($)'].sum()
total_roi = (total_net / total_cost * 100)

print("-" * 50)
print("Full Campaign Summary")
print("-" * 50)
print(f"Total customers targeted:  {len(flagged)}")
print(f"Total campaign cost:       ${total_cost:,.2f}")
print(f"Total estimated revenue saved: ${total_saved:,.2f}")
print(f"Total net value:           ${total_net:,.2f}")
print(f"Overall ROI:               {total_roi:.1f}%")

### **Visualising the ROI by the risk tiers**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# net value by tier
sns.barplot(data=roi_df, x='Risk Tier', y='Net Value ($)', 
            palette=['#0F6E56', '#1D9E75', '#9FE1CB'], ax=axes[0])
axes[0].set_title('Net value by risk tier')
axes[0].set_ylabel('Net value ($)')

# seeing the ROI percentage % by risk tier  
sns.barplot(data=roi_df, x='Risk Tier', y='ROI (%)',
            palette=['#0F6E56', '#1D9E75', '#9FE1CB'], ax=axes[1])
axes[1].set_title('ROI % by risk tier')
axes[1].set_ylabel('ROI (%)')

plt.tight_layout()
plt.savefig('../outputs/roi_by_tier.png', bbox_inches='tight')
plt.show()